# Teen Mental Health Analysis

This notebook explores the teen mental health dataset, summarizes behavioral patterns, and trains an interpretable baseline model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data' / 'raw' / 'Teen_Mental_Health_Dataset.csv')
df.head()

## Dataset shape and data types

In [ ]:
df.shape, df.dtypes

## Target balance

In [ ]:
target_dist = df['depression_label'].value_counts().sort_index()
ax = target_dist.plot(kind='bar')
ax.set_xticklabels(['No depression', 'Depression'], rotation=0)
ax.set_title('Target Distribution')
plt.show()

target_dist / len(df)

## Bucketed behavioral analysis

In [ ]:
tmp = df.copy()
tmp['sleep_bucket'] = pd.cut(tmp['sleep_hours'], bins=[0,5.99,6.99,7.99,24], labels=['<6h','6-6.9h','7-7.9h','8h+'], include_lowest=True)
tmp['social_media_bucket'] = pd.cut(tmp['daily_social_media_hours'], bins=[0,1.99,3.99,5.99,8.01], labels=['0-2h','2-4h','4-6h','6h+'], include_lowest=True)

sleep_rate = tmp.groupby('sleep_bucket', observed=False)['depression_label'].mean() * 100
social_rate = tmp.groupby('social_media_bucket', observed=False)['depression_label'].mean() * 100

display(sleep_rate.to_frame('depression_rate_pct'))
display(social_rate.to_frame('depression_rate_pct'))

## Interpretable baseline model

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=['depression_label'])
y = df['depression_label']

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
])

model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

print('ROC AUC:', round(roc_auc_score(y_test, proba), 3))
print('Average precision:', round(average_precision_score(y_test, proba), 3))

## Caveat

This dataset appears to be strongly rule-driven. Treat the modeling section as an illustration of workflow and interpretability, not as evidence of real-world clinical performance.